In [1]:
import os
import random
import numpy as np
from collections import deque

import vizdoom as vzd

import torch
import torch.nn as nn
import torch.optim as optim


# =========================
# ENV SETUP
# =========================
game = vzd.DoomGame()

config = {
    "screen_resolution": vzd.ScreenResolution.RES_160X120,
    "doom_scenario_path": os.path.join(vzd.scenarios_path, "basic.wad"),
    "doom_map": "map01",
    "available_buttons": [
        vzd.Button.MOVE_LEFT,
        vzd.Button.MOVE_RIGHT,
        vzd.Button.ATTACK,
    ],
    "episode_timeout": 300,
    "episode_start_time": 10,
    "living_reward": -0.01,
    "mode": vzd.Mode.PLAYER,
}

game.set_config(config)
game.init()


# =========================
# ACTION SPACE (BOOLEAN FORMAT)
# =========================
ACTIONS = [
    [True, False, False],   # left
    [False, True, False],   # right
    [False, False, True],   # shoot
    [False, False, False],  # noop
]


# =========================
# PREPROCESSING
# =========================
def preprocess(frame):
    # frame: (C, H, W)
    frame = frame.astype(np.float32)
    frame = np.transpose(frame, (1, 2, 0))  # HWC

    gray = frame.mean(axis=2) / 255.0
    small = gray[::2, ::2]  # downsample to ~80x60

    return small


# =========================
# FRAME STACK
# =========================
def init_stack(frame):
    return deque([frame] * 4, maxlen=4)


def get_stack(stack):
    return np.stack(stack, axis=0)


# =========================
# DQN MODEL (CNN)
# =========================
class DQN(nn.Module):
    def __init__(self, n_actions):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(4, 32, 8, stride=4),
            nn.ReLU(),
            nn.Conv2d(32, 64, 4, stride=2),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, stride=1),
            nn.ReLU(),
        )

        # 👇 dummy forward to infer shape
        with torch.no_grad():
            dummy = torch.zeros(1, 4, 80, 60)
            conv_out = self.conv(dummy)
            self.flat_size = conv_out.view(1, -1).shape[1]

        self.fc = nn.Sequential(
            nn.Linear(self.flat_size, 256),
            nn.ReLU(),
            nn.Linear(256, n_actions)
        )

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

# =========================
# TRAIN SETUP
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = DQN(len(ACTIONS)).to(device)
target_model = DQN(len(ACTIONS)).to(device)
target_model.load_state_dict(model.state_dict())

optimizer = optim.Adam(model.parameters(), lr=1e-4)

memory = deque(maxlen=10000)

gamma = 0.99
epsilon = 1.0
epsilon_min = 0.1
epsilon_decay = 0.995

batch_size = 32
target_update = 10


# =========================
# ACTION SELECT
# =========================
def select_action(state):
    global epsilon

    if random.random() < epsilon:
        return random.randint(0, len(ACTIONS) - 1)

    with torch.no_grad():
        return torch.argmax(model(state)).item()


# =========================
# TRAIN LOOP
# =========================
episodes = 300

for episode in range(episodes):
    game.new_episode()

    frame = preprocess(game.get_state().screen_buffer)
    stack = init_stack(frame)

    state = torch.tensor(get_stack(stack)).unsqueeze(0).to(device)

    total_reward = 0

    while not game.is_episode_finished():

        action = select_action(state)
        reward = game.make_action(ACTIONS[action])
        total_reward += reward

        if not game.is_episode_finished():
            next_frame = preprocess(game.get_state().screen_buffer)
            stack.append(next_frame)
            next_state = torch.tensor(get_stack(stack)).unsqueeze(0).to(device)
        else:
            next_state = None

        memory.append((state, action, reward, next_state))

        state = next_state if next_state is not None else state

        # =========================
        # TRAIN STEP
        # =========================
        if len(memory) > batch_size:

            batch = random.sample(memory, batch_size)
            s, a, r, s2 = zip(*batch)

            s = torch.cat(s)
            a = torch.tensor(a).to(device)
            r = torch.tensor(r).float().to(device)

            q = model(s).gather(1, a.unsqueeze(1)).squeeze()

            target = q.clone().detach()

            for i in range(batch_size):
                if s2[i] is not None:
                    target[i] = r[i] + gamma * target_model(s2[i]).max()
                else:
                    target[i] = r[i]

            loss = ((q - target) ** 2).mean()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    epsilon = max(epsilon_min, epsilon * epsilon_decay)

    if episode % target_update == 0:
        target_model.load_state_dict(model.state_dict())

    print(f"Episode {episode} | Reward: {total_reward:.2f} | Epsilon: {epsilon:.3f}")


game.close()

Episode 0 | Reward: 100.89 | Epsilon: 0.995
Episode 1 | Reward: -73.00 | Epsilon: 0.990
Episode 2 | Reward: 100.81 | Epsilon: 0.985
Episode 3 | Reward: 64.51 | Epsilon: 0.980
Episode 4 | Reward: 100.89 | Epsilon: 0.975
Episode 5 | Reward: -78.00 | Epsilon: 0.970
Episode 6 | Reward: 95.68 | Epsilon: 0.966
Episode 7 | Reward: 100.87 | Epsilon: 0.961
Episode 8 | Reward: -78.00 | Epsilon: 0.956
Episode 9 | Reward: 90.47 | Epsilon: 0.951
Episode 10 | Reward: 100.90 | Epsilon: 0.946
Episode 11 | Reward: 95.57 | Epsilon: 0.942
Episode 12 | Reward: 90.44 | Epsilon: 0.937
Episode 13 | Reward: 100.90 | Epsilon: 0.932
Episode 14 | Reward: 75.01 | Epsilon: 0.928
Episode 15 | Reward: 95.73 | Epsilon: 0.923
Episode 16 | Reward: 100.88 | Epsilon: 0.918
Episode 17 | Reward: 100.89 | Epsilon: 0.914
Episode 18 | Reward: -78.00 | Epsilon: 0.909
Episode 19 | Reward: 100.89 | Epsilon: 0.905
Episode 20 | Reward: 95.75 | Epsilon: 0.900
Episode 21 | Reward: 100.90 | Epsilon: 0.896
Episode 22 | Reward: 95.66 |

KeyboardInterrupt: 